In [1]:
# Installing local embedding libraries to bypass OpenAI quota issue
!pip install -q --no-warn-conflicts chromadb langchain-chroma langchain-community langchain-text-splitters sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 68.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [2]:
import os
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✓ Local setup ready. No OpenAI API Key needed now!")

/tmp/ipykernel_58/1666337794.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


✓ Local setup ready. No OpenAI API Key needed now!


In [3]:
import os

# Create directory
os.makedirs('company_docs', exist_ok=True)

# Generate sample files
sample_docs = {
    "vacation_policy.txt": "Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance.",
    "wfh_policy.txt": "Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval.",
    "maternity_policy.txt": "The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers."
}

for filename, content in sample_docs.items():
    with open(f"company_docs/{filename}", "w") as f:
        f.write(content)

print("✓ Setup complete: Sample documents generated in 'company_docs/' folder.")

✓ Setup complete: Sample documents generated in 'company_docs/' folder.


In [4]:
from sentence_transformers import SentenceTransformer

# Load a completely free, open-source embedding model
local_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    """Generate embedding for text using local Hugging Face model. Returns: List of numbers."""
    embedding = local_model.encode(text)
    return embedding.tolist()

# Test the embedding function
text = 'vacation policy'
embedding = get_embedding(text)
print(f'Embedding length: {len(embedding)} (Using local 384-dim model)')
print(f'First 5 values: {embedding[:5]}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding length: 384 (Using local 384-dim model)
First 5 values: [0.04388045147061348, 0.05842037498950958, 0.08754624426364899, -0.00025124812964349985, 0.03657165914773941]


In [5]:
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    
    dot_product = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    
    return dot_product / (norm1 * norm2)

# Test with similar and dissimilar phrases
phrases = ['vacation policy', 'time off rules', 'PTO guidelines', 'dress code requirements']
embeddings = [get_embedding(p) for p in phrases]

base_embedding = embeddings[0]
print(f'Comparing "{phrases[0]}" with:\n')

for i, phrase in enumerate(phrases[1:], start=1):
    similarity = cosine_similarity(base_embedding, embeddings[i])
    print(f'{phrase:30} Similarity: {similarity:.4f}')

Comparing "vacation policy" with:

time off rules                 Similarity: 0.2653
PTO guidelines                 Similarity: 0.1812
dress code requirements        Similarity: 0.1095


In [6]:
import chromadb
from chromadb.utils import embedding_functions

# Create ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path='./chromadb')

# Use ChromaDB's built-in default embedding function (Sentence Transformers)
# This works 100% locally and does NOT require any API key!
default_ef = embedding_functions.DefaultEmbeddingFunction()

# Get or create collection
collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=default_ef,
    metadata={'description': 'Company policy documents'}
)

print(f'Collection Name: {collection.name}')
print(f'Current Document Count: {collection.count()}')

Collection Name: company_docs
Current Document Count: 0


In [7]:
# Load text documents from the folder
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

# Split into chunks of 500 characters
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Add to ChromaDB if empty
if collection.count() == 0:
    collection.add(
        documents=[chunk.page_content for chunk in chunks],
        ids=[f'doc_{i}' for i in range(len(chunks))],
        metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
    )
    print(f'✓ Successfully added {len(chunks)} chunks to ChromaDB.')
else:
    print(f'Collection already contains {collection.count()} documents. Skipping insertion.')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 38.8MiB/s]


✓ Successfully added 3 chunks to ChromaDB.


In [8]:
def vector_search(query, n_results=3):
    """Search ChromaDB collection using semantic similarity."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test queries
test_queries = ['time off policy', 'WFH guidelines', 'maternity leave']

for query in test_queries:
    print(f'\n{"="*60}')
    print(f'Query: {query}')
    print(f'{"="*60}')
    
    results = vector_search(query, n_results=2)
    
    for i, doc in enumerate(results['documents'][0]):
        distance = results['distances'][0][i]
        print(f'\nResult {i+1} (Distance/Score: {distance:.4f}):')
        print(doc[:200] + '...')


Query: time off policy

Result 1 (Distance/Score: 1.0252):
Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....

Result 2 (Distance/Score: 1.3234):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: WFH guidelines

Result 1 (Distance/Score: 1.2676):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....

Result 2 (Distance/Score: 1.8423):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Query: maternity leave

Result 1 (Distance/Score: 0.6504):
The parental leave policy provides 12 weeks of fully paid maternity leave for new mothers....

Result 2 (Distance/Score: 1.5459):
Our remote work and WFH guidelines allow employees to work from home up to 3 days a week with manager approval....


In [9]:
# Install required package
!pip install -q langchain-openai langchain-community langchain

# Import
from langchain_openai import ChatOpenAI
import os

# Set API Key
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM
try:
    llm = ChatOpenAI(
        model="gpt-3.5-turbo",
        temperature=0
    )
    print("LLM Initialized Successfully!")

except Exception as e:
    print("Error:", e)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.3 MB/s eta 0:00:00
LLM Initialized Successfully!


In [10]:
def keyword_search(query, chunks, top_k=3):
    """Simple keyword frequency counter."""
    scored = []
    query_lower = query.lower()
    words = query_lower.split()
    
    for chunk in chunks:
        score = sum(chunk.page_content.lower().count(word) for word in words)
        if score > 0:
            scored.append((score, chunk))
            
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for s, c in scored[:top_k]]

# Run Test
query = 'PTO policy'

print('--- KEYWORD SEARCH ---')
kw_results = keyword_search(query, chunks, top_k=2)
print(f'Found: {len(kw_results)} results')

print('\n--- SEMANTIC SEARCH ---')
sem_results = vector_search(query, n_results=2)
print(f'Found: {len(sem_results["documents"][0])} results')
if len(sem_results["documents"][0]) > 0:
    print(f'Top result: {sem_results["documents"][0][0][:120]}...')

--- KEYWORD SEARCH ---
Found: 2 results

--- SEMANTIC SEARCH ---
Found: 2 results
Top result: Employees get 20 days of paid time off (PTO) annually. All vacation requests must be submitted two weeks in advance....
